## Question 1

### ITMLA2 Project: Question 1 - Predictive Maintenance Feature Selection
#### Student: Sekwanele Njoko (EDUV9294617)
#### Module: Machine Learning Algorithms (ITMLA2-44)

1. Business Context & Objective
A municipal water utility operates a large fleet of pumping assets. Unexpected mechanical failures interrupt supply and increase costs. The objective of this notebook is to prepare historical telemetry data (2,149 records) and identify the most informative predictors for a pump asset requiring a major service (Major_Service_Required).

2. Methodology
This notebook covers the following steps:

Data Audit & Preprocessing: Handling missing values, removing non-predictive identifiers (Asset_ID), encoding categorical variables (LabelEncoder), and scaling numerical features (StandardScaler).

Filter Method (SelectKBest): Using the ANOVA F-value (f_classif) to retain the 8 strongest predictors based on individual statistical relationships.

Wrapper Method (RFE): Applying Recursive Feature Elimination with Logistic Regression to select 8 predictors based on how well they perform together in a model.

Comparative Analysis: Comparing the results of both methods to identify robust, agreed-upon features versus method-specific features.

3. Outputs
The outputs of this notebook (selected feature tables and scores) are used to write the methodology and analysis section of the final project report.

In [8]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression

# 1. Load the dataset
df = pd.read_csv(r"C:\Users\KHABAZELA\OneDrive\Desktop\ITMLA2_Project\data\machine_service_risk.csv")

In [7]:
df.head(10)

,Asset_ID,Equipment_Age_Months,Service_Contract,Operating_Zone,Maintenance_Plan,Average_Load_Tonnes,Dust_Exposure_Level,Weekly_Idle_Hours,Daily_Operating_Hours,Lubrication_Quality_Score,...,Mechanical_Condition_Score,Error_Log_Alert,Irregular_Vibration_Alert,Maintenance_Compliance_Score,Sensor_Mismatch_Flag,Calibration_Drift_Flag,Noise_Anomaly_Flag,Start_Up_Failure_Flag,Intermittent_Shutdown,Major_Service_Required
0,MX-10001,73,Extended,Central,Enhanced,22.927749,Low,13.297218,6.327112,1.347214,...,6.518877,0,0,1.725883,0,0,0,1,No,0
1,MX-10002,89,Extended,Central,NaN,26.827681,Low,4.542524,7.619885,0.518767,...,7.118696,0,0,2.592424,0,0,0,0,Yes,0
2,MX-10003,73,Extended,Remote,Basic,17.795882,Low,19.555085,7.844988,1.826335,...,5.895077,0,0,7.119548,0,1,0,1,No,0
3,MX-10004,74,Standard,Central,Basic,33.800817,High,12.209266,8.428001,7.435604,...,8.965106,0,1,6.481226,0,0,0,0,No,0
4,MX-10005,89,Extended,Central,NaN,20.716974,Low,18.454356,6.310461,0.795498,...,6.045039,0,0,0.014691,0,0,1,1,No,0
5,MX-10006,86,Standard,North,Basic,30.626886,Low,4.140144,0.211062,1.584922,...,5.510144,0,0,9.015686,1,0,0,0,No,0
6,MX-10007,68,Extended,Remote,Enhanced,38.387622,High,0.646047,9.257695,5.897388,...,6.062124,0,0,9.236328,0,0,0,0,Yes,0
7,MX-10008,75,Extended,Central,Basic,18.776009,Low,13.723826,4.649451,8.341903,...,3.401374,0,0,4.517248,1,0,0,0,Yes,1
8,MX-10009,72,Standard,North,NaN,27.833188,Low,12.167848,1.531360,6.736882,...,7.396061,0,1,0.756232,0,0,1,0,No,0
9,MX-10010,87,Extended,Central,NaN,35.456302,High,16.028688,6.440773,8.086019,...,1.148904,0,1,4.554394,0,0,0,0,No,0


#### 1.1 Data Audit and Preprocessing

In [3]:
# --- 1.1 DATA PREPARATION ---

# Initial Data Audit
print("--- Initial Data Audit ---")
print(df.info())
print("\nMissing values before treatment:\n", df.isnull().sum())

# 1. Handle Missing Values (Drop rows with missing data)
# We do this to ensure the model trains on complete records
df_clean = df.dropna()

# 2. Remove Non-Predictive Identifiers
# Asset_ID has no predictive power
df_clean = df_clean.drop(columns=['Asset_ID'])

# 3. Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
categorical_cols = ['Service_Contract', 'Operating_Zone', 'Maintenance_Plan', 'Dust_Exposure_Level', 'Intermittent_Shutdown']

for col in categorical_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

# 4. Separate Features (X) and Target (y)
X = df_clean.drop(columns=['Major_Service_Required'])
y = df_clean['Major_Service_Required']

# 5. Feature Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\nData preparation complete. Ready for feature selection.")

--- Initial Data Audit ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Asset_ID                       2149 non-null   object 
 1   Equipment_Age_Months           2149 non-null   int64  
 2   Service_Contract               2149 non-null   object 
 3   Operating_Zone                 2149 non-null   object 
 4   Maintenance_Plan               1703 non-null   object 
 5   Average_Load_Tonnes            2149 non-null   float64
 6   Dust_Exposure_Level            2149 non-null   object 
 7   Weekly_Idle_Hours              2149 non-null   float64
 8   Daily_Operating_Hours          2149 non-null   float64
 9   Lubrication_Quality_Score      2149 non-null   float64
 10  Cooling_Efficiency_Score       2149 non-null   float64
 11  Similar_Model_Failure_History  2149 non-null   int64  
 12  Hydraulic_Fault_Histo

#### 1.2 SelectKBest Feature Selection

In [4]:
# --- 1.2 SELECTKBEST ---
# Apply SelectKBest with f_classif (ANOVA F-value for classification)
selector = SelectKBest(score_func=f_classif, k=8)
X_kbest = selector.fit_transform(X_scaled, y)

# Create a DataFrame to display the results neatly
kbest_results = pd.DataFrame({
    'Feature': X.columns[selector.get_support()],
    'Score': selector.scores_[selector.get_support()]
}).sort_values(by='Score', ascending=False)

print("--- SelectKBest (Top 8) ---")
print(kbest_results.to_string(index=False))

--- SelectKBest (Top 8) ---
                      Feature      Score
   Mechanical_Condition_Score 239.806807
 Maintenance_Compliance_Score 199.582215
              Error_Log_Alert 186.084761
    Irregular_Vibration_Alert  87.400848
          Sensor_Health_Score  84.858494
        Coolant_Temperature_C   5.099237
     Cooling_Efficiency_Score   3.583793
Average_Bearing_Temperature_C   3.311945


#### 1.3 Recursive Feature Elimination (RFE)

In [5]:
# --- 1.3 RFE (Recursive Feature Elimination) ---
# Apply RFE using Logistic Regression
estimator = LogisticRegression(max_iter=1000)
rfe = RFE(estimator=estimator, n_features_to_select=8)
X_rfe = rfe.fit_transform(X_scaled, y)

# Get the selected feature names
rfe_features = X.columns[rfe.support_]

print("--- RFE (Top 8) ---")
for feature in rfe_features:
    print(feature)

--- RFE (Top 8) ---
Impact_Damage_History
Average_Bearing_Temperature_C
Coolant_Temperature_C
Sensor_Health_Score
Mechanical_Condition_Score
Error_Log_Alert
Irregular_Vibration_Alert
Maintenance_Compliance_Score


#### 1.4 Comparative Analysis of Feature Sets

In [6]:
# --- 1.4 COMPARISON ---
print("--- Feature Comparison ---")
common_features = set(kbest_results['Feature']).intersection(set(rfe_features))
print(f"Common Predictors: {list(common_features)}")

unique_kbest = set(kbest_results['Feature']) - set(rfe_features)
unique_rfe = set(rfe_features) - set(kbest_results['Feature'])
print(f"Unique to SelectKBest: {list(unique_kbest)}")
print(f"Unique to RFE: {list(unique_rfe)}")

--- Feature Comparison ---
Common Predictors: ['Coolant_Temperature_C', 'Irregular_Vibration_Alert', 'Sensor_Health_Score', 'Error_Log_Alert', 'Mechanical_Condition_Score', 'Average_Bearing_Temperature_C', 'Maintenance_Compliance_Score']
Unique to SelectKBest: ['Cooling_Efficiency_Score']
Unique to RFE: ['Impact_Damage_History']


### Cleaned data: `final_features`

In [ ]:
# 1. Create a new DataFrame with only the RFE features and the target variable
# Adding .copy() fixes the SettingWithCopyWarning
final_features = X_scaled[rfe_features].copy()

# Now it's safe to add the target variable
final_features['Major_Service_Required'] = y

# 2. Save  to  data folder
os.makedirs('../data', exist_ok=True)
final_features.to_csv('../data/cleaned_rfe_data.csv', index=False)

print("Clean dataset saved successfully to ../data/cleaned_rfe_data.csv")
print("Shape of saved data:", final_features.shape)

Clean dataset saved successfully to ../data/cleaned_rfe_data.csv
Shape of saved data: (1703, 9)
